# Massive API Practice - Options

In [19]:
import yaml
from pathlib import Path
from massive import RESTClient
import pandas as pd

CONFIG_PATH = Path("../data_preparation/data_apiKey/config.yaml")
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

API_KEY = config["keys"]["polygon_api_key"]
client = RESTClient(api_key=API_KEY)

## Options Contracts

In [ ]:
# list all options contracts for AAPL
contracts = []
for c in client.list_options_contracts("AAPL"):
    contracts.append(c)

print(f"Total contracts: {len(contracts)}")
contracts

In [ ]:
# get a single contract by ticker
contract = client.get_options_contract("O:AAPL260101C00190000")
contract

## Options Chain Snapshot

In [ ]:
options_chain = []
for o in client.list_snapshot_options_chain(
    "AAPL",
    params={
        "expiration_date.gte": "2026-03-01",
        "expiration_date.lte": "2026-06-30",
        "strike_price.gte": 180,
        "strike_price.lte": 250,
    },
):
    options_chain.append(o)

print(f"Chain length: {len(options_chain)}")
options_chain[:5]

In [17]:
# snapshot for a specific option contract
snapshot = client.get_snapshot_option("AAPL", "O:AAPL260618C00180000")
snapshot

OptionContractSnapshot(break_even_price=260.95, day=DayOptionContractSnapshot(change=0, change_percent=0, close=79.66, high=79.66, last_updated=1772830800000000000, low=79.66, open=79.66, previous_close=79.66, volume=1, vwap=79.66), details=OptionDetails(contract_type='call', exercise_style='american', expiration_date='2026-06-18', shares_per_contract=100, strike_price=180, ticker='O:AAPL260618C00180000'), greeks=Greeks(delta=0.9242623058493726, gamma=0.0019708919539114926, theta=-0.06261641482277651, vega=0.1899979558123886), implied_volatility=0.5275981698819663, last_quote=None, last_trade=None, open_interest=1524, underlying_asset=UnderlyingAsset(change_to_break_even=4.3, last_updated=1772845199594173594, price=256.65, value=None, ticker='AAPL', timeframe='REAL-TIME'), fair_market_value=None)

## Aggregates / OHLCV Bars

In [18]:
aggs = []
for a in client.list_aggs(
    "O:SPY261218C00600000",
    1,
    "day",
    "2025-09-01",
    "2026-03-01",
    limit=50000,
):
    aggs.append(a)

print(f"Total bars: {len(aggs)}")
aggs[:5]

Total bars: 124


[Agg(open=87.45, high=89.9, low=87.45, close=89.9, volume=16, vwap=88.4513, timestamp=1756785600000, transactions=6, otc=None),
 Agg(open=92.94, high=93.38, low=92.07, close=92.16, volume=24, vwap=92.555, timestamp=1756872000000, transactions=15, otc=None),
 Agg(open=94.4, high=96.2, low=94.4, close=96.2, volume=10, vwap=95.143, timestamp=1756958400000, transactions=10, otc=None),
 Agg(open=98, high=98, low=94.15, close=94.97, volume=13, vwap=95.5023, timestamp=1757044800000, transactions=6, otc=None),
 Agg(open=95.8, high=96.54, low=95.62, close=95.62, volume=9, vwap=95.9911, timestamp=1757304000000, transactions=5, otc=None)]

In [20]:
df_aggs = pd.DataFrame([{
    "timestamp": a.timestamp,
    "open": a.open,
    "high": a.high,
    "low": a.low,
    "close": a.close,
    "volume": a.volume,
} for a in aggs])

df_aggs["date"] = pd.to_datetime(df_aggs["timestamp"], unit="ms")
df_aggs.set_index("date", inplace=True)
df_aggs.drop(columns=["timestamp"], inplace=True)
df_aggs.head(10)

,open,high,low,close,volume
date,,,,,
2025-09-02 04:00:00,87.45,89.90,87.45,89.90,16
2025-09-03 04:00:00,92.94,93.38,92.07,92.16,24
2025-09-04 04:00:00,94.40,96.20,94.40,96.20,10
2025-09-05 04:00:00,98.00,98.00,94.15,94.97,13
2025-09-08 04:00:00,95.80,96.54,95.62,95.62,9
2025-09-09 04:00:00,96.07,97.35,96.07,97.35,13
2025-09-10 04:00:00,100.07,100.07,99.30,99.30,6
2025-09-11 04:00:00,102.23,104.20,102.00,104.20,31
2025-09-12 04:00:00,103.83,104.38,103.65,103.65,14


## Daily Open/Close & Previous Close

In [23]:
daily_oc = client.get_daily_open_close_agg("O:SPY261218C00600000", "2026-01-06")
daily_oc

DailyOpenCloseAgg(after_hours=125.71, close=125.71, from_='2026-01-06', high=125.71, low=122.2, open=122.5, pre_market=122.5, status='OK', symbol='O:SPY261218C00600000', volume=56, otc=None)

In [25]:
prev_close = client.get_previous_close_agg("O:SPY261218C00600000")
prev_close

[PreviousCloseAgg(ticker='O:SPY261218C00600000', close=106.77, high=109.27, low=106.77, open=107.61, timestamp=1772830800000, volume=8, vwap=108.2225)]

## Trades & Quotes

In [ ]:
last_trade = client.get_last_trade("O:TSLA260619C00250000")
last_trade

In [ ]:
trades = []
for t in client.list_trades("O:TSLA260619C00250000", limit=50000):
    trades.append(t)

print(f"Total trades: {len(trades)}")
trades[:3]

In [ ]:
quotes = []
for q in client.list_quotes("O:SPY261218C00600000", limit=50000):
    quotes.append(q)

print(f"Total quotes: {len(quotes)}")
quotes[:3]

## Technical Indicators

In [28]:
sma = client.get_sma(
    ticker="O:SPY261218C00600000",
    timespan="day", window=20, series_type="close"
)
sma

SingleIndicatorResults(values=[IndicatorValue(timestamp=1772773200000, value=117.6425), IndicatorValue(timestamp=1772686800000, value=117.999), IndicatorValue(timestamp=1772600400000, value=118.4115), IndicatorValue(timestamp=1772514000000, value=118.542), IndicatorValue(timestamp=1772427600000, value=119.228), IndicatorValue(timestamp=1772168400000, value=119.579), IndicatorValue(timestamp=1772082000000, value=119.94200000000001), IndicatorValue(timestamp=1771995600000, value=120.30499999999999), IndicatorValue(timestamp=1771909200000, value=120.548), IndicatorValue(timestamp=1771822800000, value=120.91650000000001)], underlying=IndicatorUnderlying(url='https://api.polygon.io/v2/aggs/ticker/O:SPY261218C00600000/range/1/day/1401681600000/1772945257466?limit=102&sort=desc', aggregates=[]))

In [34]:
ema = client.get_ema(
    ticker="O:SPY261218C00600000",
    timespan="day", window=50, series_type="close"
)
ema

SingleIndicatorResults(values=[IndicatorValue(timestamp=1772773200000, value=118.54114225698127), IndicatorValue(timestamp=1772686800000, value=119.0215970429805), IndicatorValue(timestamp=1772600400000, value=119.40492753453073), IndicatorValue(timestamp=1772514000000, value=119.50839396451157), IndicatorValue(timestamp=1772427600000, value=119.75771616714468), IndicatorValue(timestamp=1772168400000, value=119.85231682702813), IndicatorValue(timestamp=1772082000000, value=119.96873792200887), IndicatorValue(timestamp=1771995600000, value=119.98582926576434), IndicatorValue(timestamp=1771909200000, value=119.85382229702003), IndicatorValue(timestamp=1771822800000, value=119.92214157444941)], underlying=IndicatorUnderlying(url='https://api.polygon.io/v2/aggs/ticker/O:SPY261218C00600000/range/1/day/1401681600000/1772946635768?limit=235&sort=desc', aggregates=[]))

In [35]:
rsi = client.get_rsi(
    ticker="O:SPY261218C00600000",
    timespan="day", window=14, series_type="close"
)
rsi

SingleIndicatorResults(values=[IndicatorValue(timestamp=1772773200000, value=37.96714796394526), IndicatorValue(timestamp=1772686800000, value=40.3767351646086), IndicatorValue(timestamp=1772600400000, value=47.45648841556375), IndicatorValue(timestamp=1772514000000, value=43.009171058730146), IndicatorValue(timestamp=1772427600000, value=47.34113573750964), IndicatorValue(timestamp=1772168400000, value=46.79922416129284), IndicatorValue(timestamp=1772082000000, value=49.54295608951927), IndicatorValue(timestamp=1771995600000, value=53.75468642593731), IndicatorValue(timestamp=1771909200000, value=48.1317957178967), IndicatorValue(timestamp=1771822800000, value=42.48889345710815)], underlying=IndicatorUnderlying(url='https://api.polygon.io/v2/aggs/ticker/O:SPY261218C00600000/range/1/day/1401681600000/1772946637328?limit=75&sort=desc', aggregates=[]))

In [36]:
macd = client.get_macd(
    ticker="O:SPY261218C00600000",
    timespan="day",
    short_window=12,
    long_window=26,
    signal_window=9,
    series_type="close",
)
macd

MACDIndicatorResults(values=[MACDIndicatorValue(timestamp=1772773200000, value=-2.5001388822211084, signal=-1.5113949647674558, histogram=-0.9887439174536525), MACDIndicatorValue(timestamp=1772686800000, value=-1.8685355353913167, signal=-1.2642089854040426, histogram=-0.6043265499872741), MACDIndicatorValue(timestamp=1772600400000, value=-1.326401292195584, signal=-1.113127347907224, histogram=-0.21327394428836), MACDIndicatorValue(timestamp=1772514000000, value=-1.3523113848119976, signal=-1.0598088618351338, histogram=-0.29250252297686385), MACDIndicatorValue(timestamp=1772427600000, value=-1.0124030101542303, signal=-0.9866832310909177, histogram=-0.025719779063312576), MACDIndicatorValue(timestamp=1772168400000, value=-0.975175913625165, signal=-0.9802532863250895, histogram=0.005077372699924476), MACDIndicatorValue(timestamp=1772082000000, value=-0.8686762244026056, signal=-0.9815226295000706, histogram=0.11284640509746502), MACDIndicatorValue(timestamp=1771995600000, value=-0.97

## Exchanges & Conditions

In [29]:
exchanges = client.get_exchanges("options")
for ex in exchanges:
    print(f"{ex.asset_class:<15}{ex.name} ({ex.operating_mic})")

options        NYSE American Options (XNYS)
options        Boston Options Exchange (XBOX)
options        Chicago Board Options Exchange (XCBO)
options        MIAX Emerald, LLC (MIHI)
options        Cboe EDGX Options (XCBO)
options        Nasdaq Global Markets Exchange Group (GEMX)
options        International Securities Exchange, LLC (XISX)
options        Nasdaq MRX Options Exchange (XISX)
options        Nasdaq GMNI Options Exchange (XISX)
options        MIAX International Securities Exchange, LLC (MIHI)
options        NYSE Arca, Inc. - Options (XNYS)
options        Options Price Reporting Authority (OPRA)
options        MIAX Pearl, LLC - Options (MIHI)
options        Nasdaq Options Market (XNAS)
options        MIAX SAPPHIRE, LLC (MIHI)
options        Nasdaq BX - Options (XNAS)
options        Members Options Exchange (MEMX)
options        Cboe C2 Options Exchange (XCBO)
options        Nasdaq Philadelphia Exchange, LLC - Options (XNAS)
options        Cboe BZX Options Exchange (XCBO)


In [30]:
conditions = []
for c in client.list_conditions("options", limit=1000):
    conditions.append(c)
conditions[:10]

[Condition(abbreviation='CANC', asset_class='options', data_types=['trade'], description='Transaction previously reported (other than as the last or opening report for the particular option contract) is now to be canceled.', exchange=None, id=201, legacy=None, name='Canceled', sip_mapping=SipMapping(CTA=None, OPRA='A', UTP=None), type='sale_condition', update_rules=UpdateRules(consolidated=Consolidated(updates_high_low=False, updates_open_close=False, updates_volume=False), market_center=MarketCenter(updates_high_low=False, updates_open_close=False, updates_volume=False))),
 Condition(abbreviation='OSEQ', asset_class='options', data_types=['trade'], description='Transaction is being reported late and is out of sequence; i.e., later transactions have been reported for the particular option contract.', exchange=None, id=202, legacy=None, name='Late and Out Of Sequence', sip_mapping=SipMapping(CTA=None, OPRA='B', UTP=None), type='sale_condition', update_rules=UpdateRules(consolidated=Cons

## Options Chain -> DataFrame

In [31]:
chain = []
for o in client.list_snapshot_options_chain(
    "SPY",
    params={
        "expiration_date.gte": "2026-06-01",
        "expiration_date.lte": "2026-06-30",
        "strike_price.gte": 550,
        "strike_price.lte": 650,
    },
):
    chain.append(o)

rows = []
for o in chain:
    rows.append({
        "ticker": o.details.ticker if o.details else None,
        "contract_type": o.details.contract_type if o.details else None,
        "strike": o.details.strike_price if o.details else None,
        "expiration": o.details.expiration_date if o.details else None,
        "last_price": o.day.close if o.day else None,
        "volume": o.day.volume if o.day else None,
        "open_interest": o.open_interest,
        "iv": o.implied_volatility,
    })

df_chain = pd.DataFrame(rows)
df_chain.head(20)

,ticker,contract_type,strike,expiration,last_price,volume,open_interest,iv
0,O:SPY260618C00550000,call,550,2026-06-18,132.00,3.0,12430,0.344148
1,O:SPY260618C00555000,call,555,2026-06-18,129.18,4.0,350,0.337179
2,O:SPY260618C00560000,call,560,2026-06-18,124.23,1.0,1899,0.329387
3,O:SPY260618C00565000,call,565,2026-06-18,122.19,12.0,667,0.321925
4,O:SPY260618C00570000,call,570,2026-06-18,117.27,17.0,5156,0.317703
5,O:SPY260618C00575000,call,575,2026-06-18,115.90,3.0,3139,0.311722
6,O:SPY260618C00580000,call,580,2026-06-18,111.02,3.0,1858,0.304070
7,O:SPY260618C00585000,call,585,2026-06-18,98.99,2.0,567,0.300030
8,O:SPY260618C00590000,call,590,2026-06-18,98.55,4.0,2064,0.294911
9,O:SPY260618C00595000,call,595,2026-06-18,98.47,3.0,1082,0.290505


In [32]:
calls = df_chain[df_chain["contract_type"] == "call"].sort_values("strike")
puts = df_chain[df_chain["contract_type"] == "put"].sort_values("strike")

print(f"Calls: {len(calls)}, Puts: {len(puts)}")
calls.head(10)

Calls: 115, Puts: 115


,ticker,contract_type,strike,expiration,last_price,volume,open_interest,iv
0,O:SPY260618C00550000,call,550,2026-06-18,132.00,3.0,12430,0.344148
42,O:SPY260630C00550000,call,550,2026-06-30,145.68,3.0,112,0.318979
43,O:SPY260630C00554000,call,554,2026-06-30,148.52,1.0,3,0.317977
44,O:SPY260630C00555000,call,555,2026-06-30,134.29,37.0,85,0.315962
1,O:SPY260618C00555000,call,555,2026-06-18,129.18,4.0,350,0.337179
45,O:SPY260630C00556000,call,556,2026-06-30,132.14,3.0,3,0.319807
46,O:SPY260630C00557000,call,557,2026-06-30,NaN,NaN,0,0.313876
47,O:SPY260630C00558000,call,558,2026-06-30,131.55,2.0,6,0.312574
48,O:SPY260630C00559000,call,559,2026-06-30,126.60,2.0,6,0.311275
49,O:SPY260630C00560000,call,560,2026-06-30,126.40,2.0,80,0.305541


In [33]:
puts.head(10)

,ticker,contract_type,strike,expiration,last_price,volume,open_interest,iv
21,O:SPY260618P00550000,put,550,2026-06-18,6.80,6463.0,64112,0.342971
136,O:SPY260630P00550000,put,550,2026-06-30,7.40,67.0,3570,0.332832
137,O:SPY260630P00554000,put,554,2026-06-30,7.42,12.0,99,0.327519
138,O:SPY260630P00555000,put,555,2026-06-30,7.79,11.0,7624,0.326213
22,O:SPY260618P00555000,put,555,2026-06-18,6.37,135.0,3158,0.336204
139,O:SPY260630P00556000,put,556,2026-06-30,5.52,7.0,58,0.324910
140,O:SPY260630P00557000,put,557,2026-06-30,6.89,1.0,9,0.323610
141,O:SPY260630P00558000,put,558,2026-06-30,5.11,2.0,22,0.322314
142,O:SPY260630P00559000,put,559,2026-06-30,5.04,1.0,14,0.321078
143,O:SPY260630P00560000,put,560,2026-06-30,8.19,20.0,290,0.319845
